# Sentinel Grid - Honeypot Log Piepline 
***This is currently a working basic structure for our pipeline. It is currenlty a skeleton until we collect real honeypot data where I will adjust as needed.***

### Expected Inputs
This notebook is the data preprocessing pipeline for Sentinel Grid's honepot data. It expects Cowrie JSON logs and the purpose it to conver the raw Cowrie logs to a clean session level feature table that will be used for anomaly detection, attacker clustering, behavioral profiling, and supervised classification.

Common Cowrie Even Types:
- `cowrie.session.connect`, `cowrie.session.closed`
- `cowrie.login.success`, `cowrie.login.failed`
- `cowrie.command.input` (commands typed)
- `cowrie.client.version`

Inputs/Fields:
- `eventid`
- `timestamp`
- `session`
- `src_ip`, `dst_ip`, `dst_port`, `protocol`
- `username`, `password` 
- `input`

Optional Inputs/Fields:
- Geographic Location map to IP
- country/city/lat/lon

### Outputs
1) `events_df`: event level tables
2) `session_df`: session level tables 
3) `features_df`: machine learning ready features 
3) Info needed for  visualization 

Analogy:
- `events_df` -> Every time scanned at checkout
- `session_df` -> One full customer shopping visit
- `features_df` -> Customer profile used to predict future purchases


In [ ]:
from __future__ import annotations
import os
import json
import math
import re
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 

#edit path/directory here:
COWIRE_LOGS_PATH = Path("../data/cowirelogs/")
OUTPUT_DIR= Path("../data/outputs/")
OUTPUT_DIR.mkdir(parents= True, exist_ok= True)

#change to True if we want to use geoip
GEOIP= False
GEOIP_MMDB= Path("../data/geoip/")

#if we want reeproducibility in samples and plots:
RANDOM_SEED= 42
np.random.seed(RANDOM_SEED)

#show max 200 columns if printing dataframe before truncating  
pd.set_option("display.max_columns",200)

#### Load Cowrie JSON logs from file or directoy
This step takes raw Cowrie honeypot logs and converts them into a DataFrame which is a 2D table structure to store and manipulate data. 
- Checks if path is a directory or a single file 
- Reads logs line by line and parses JSON events and skips malformed entries
- Outputs: `events_df` one row per honepot event 

In [ ]:
#reads json log line by line 
#opens in read mode, iterates through each line to remove whitespace and parse line
def iter_json_lines(path:Path) ->Iterable[Dict[str, Any]]:
    with path.open("r", encoding= "utf-8") as f:
        for line in f:
            line= line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                #skip malformed lines 
                continue 

#load cowrie json logs from a file or directory into DataFrame
def load_cowrie_logs(path: Path) -> pd.DataFrame:
    #checks if path is a file or directory 
    paths: List[Path]= []
    if path.is_dir():
        paths= sorted(list(path.glob("cowrie.json"))) or sorted(list(path.glob("*.json")))
    else:
        paths=[path]

    #list to store json rows 
    rows= List[Dict[str, Any]]= []
    for p in paths:
        for obj in iter_json_lines(p):
            obj["_source_file"]= p.name
            rows.append(obj)
    
    df= pd.json_normalize(rows)
    if df.empty:
        raise ValueError(f"No Cowrie JSON events found at: {path}")
    return df

events_df= load_cowrie_logs(COWIRE_LOGS_PATH)
#print and display first 5 rows of data frame
print("Loaded Events:", events_df.shape)
events_df.head(5)

#### Standardizing DataFrame
Cleans and normalizes Cowrie honeypot DataFrame ti check if there are missing or inconsistent columns. It prepares logs for reliable analysis.  
- Verifies certain columns exists
- Converts timestamps to date time format
- Normalizes numeric fields and handles missing or inconsistent values

In [ ]:
def ensure_col(df: pd.DataFrame, col:str, default= None) ->pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series([default]*len(df), index= df.index)

#standardize expected columns
events_df["eventid"]= ensure_col(events_df, "eventid", default= None)
events_df["session"]= ensure_col(events_df, "session", default= None)

#timestamp
events_df["timestamp"]= pd.to_datetime(ensure_col(events_df, "timestamp", default= None), errors= "coerce", utc=True)

#ip fields
events_df["src_ip"]= ensure_col(events_df, "src_ip", default=None)
events_df["dst_ip"]= ensure_col(events_df, "dst_ip", default=None)
events_df["dst_port"]= pd.to_numeric(ensure_col(events_df, "dst_port", default=np.nan), errors="coerce")

#user, pass, command inputs
events_df["username"]= ensure_col(events_df, "username", default=None)
events_df["password"]= ensure_col(events_df, "password", default=None)
events_df["command"]= ensure_col(events_df, "input", defaul= None)

#protocol fields may vary 
events_df["protocol"]= ensure_col(events_df, "protocol", default= None)
if events_df["protocol"].isna().all():
    events_df["protocol"]= ensure_col(events_df, "system.transport", default= None)
#view 10 rows
events_df[["timestamp", "eventid", "session", "src_ip", "dst_ip", "dst_port", "username", "passwprd", "command","protocol"]].head(10)

print("Event Types:")
print(events_df["eventid"].value_counts(), "\n")
print("Missing Session IDs:", events_df["session"].isna().sum())
print("Missing Timestamps", events_df["timestamp"].isna().sum())
print("Unique Sessions:", events_df["session"].nunique())

#### Session Boundaries
Groups events by session ID and computes session duration using timestamps
- `session_start` is the earliest event timestamp
- `session_end` is the latest event timestamp
- `duration` is the total interaction time in seconds 

In [ ]:
#remove events that are missing session id or not valid
sess_events= events_df.dropna(subset=["session"]).copy()

#group events by ssession id and compute start and end times 
session_times=(
    sess_events.groupby("session")["timestamp"]
    .agg(session_start= "min", session_end= "max")
    .reset_index()

)

#compute durations and clip negatives to 0
session_times["duration"]= (session_times["session_end"]-session_times["session_start"]).dt.total_seconds()
session_times["duration"]= session_times["duration"].clip(lower=0)

session_times.head()

### Aggregate Session Level Fields
Group data by session and compute summaries 
- Extract ip, destination port, protocol
- Extract login activities of successes, failures, unique usernames and passwords
- Extract comman activity 

In [ ]:
def first_nonnull(series: pd.Series):
    s= series.dropna()
    return s.iloc[0] if len(s) else None

#session data
session_net= (
    sess_events.groupby("session").agg(
        src_ip= ("src_ip", first_nonnull),
        dst_ip= ("dst_ip", first_nonnull),
        dst_port= ("dst_port", first_nonnull),
        protocol= ("protocol", first_nonnull),
        user_first= ("username", first_nonnull),
    ).reset_index()
)

#success/fail login attempts
login_success= sess_events["eventid"].astype(str).eq("cowrie.login.success")
login_fail= sess_events["eventid"].astype(str).eq("cowrie.login.failed")

login_agg= (
    sess_events.groupby("session").apply(lambda g: pd.Series({
        "login_success": int((g["eventid"]=="cowrie.login.success").sum()),
        "login_fail": int((g["eventid"]=="cowrie.login.failed").sum()),
        "unique_users": g["username"].dropna().nunique(),
        "unique_pass": g["password"].dropna().nuniqu(),
    })).reset_index()
)

#commands
is_cmd= sess_events["eventid"].astype(str).eq("cowrie.command.input")
cmd_df= sess_events[is_cmd & sess_events["command"].notna()].copy()

cmd_agg= (
    cmd_df.groupby("session").agg(
        cmd_count= ("command", "count"),
        unique_cmds= ("command", pd.Series.nunique),
        first_cmd= ("command", first_nonnull),
        last_cmd= ("command", lambda s: s.dropna().iloc[-1] if s.dropna().shape[0] else None),
    ).reset_index()
)

#merge session tables
session_df= (session_times.merge(session_net, on= "session", how= "left")
             .merge(login_agg, on= "session", how= "left")
             .merge(cmd_agg, on= "session", how= "left")
             )
#fill empty or missing counts wiht 0
for c in ["login_success", "login_fail",  "cmd_count", "unique_cmds", "unique_users", "unique_pass"]:
    if c in session_df.columns:
        session_df[c]= session_df[c].fillna(0).astype(int)
#print first 10 
session_df.head(10)

### Derive Features
Password entropy, rates, bot likeness

### Shannon Entropy 
Shannon entropy measures password strength in bits, representing min number of guesses needed to guess it.

The Shannon entropy is defined as:
$$
H = -\sum_{i=1}^{n} p(x_i)\log_2 p(x_i)
$$

where:
- $p(x_i)$ is the probability of character $x_i$ in the password  
- $n$ is the number of unique characters  

This helps us measure how unpredictable a password string is.

- Higher entropy → more randomness → more human-like
- Medium entropy → common human style passwords
- Lower entropy → repetitive/simple → more bot-like

<u> Character distribution entropy:</u> Instead of estimating entropy from the allowed character pool, we compute it from the actual characters used. This better captures repetition patterns, structured guesses, and dictionary based attacks.

In our pipeline, entropy is computed per password attempt and can be aggregated per session (mean entropy or maximum entropy) to characterize attacker behavior.


In [ ]:
def shannon_entropy(s:str) ->float:
    #handles missing values or nons trings
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return 0.0
    
    s=str(s)
    if len(s) == 0:
        return 0.0
        
    #counts how many times each character appears
    #stored in dict - ex: "aab1"-> {'a':2, 'b':1, '1':1}
    counts= {}
    for ch in s:
        counts[ch]= counts.get(ch, 0)+1
    
    #compute probability by dividing count by total length of string
    #ex: [2,1,1]/4 -> [0.5, 0.25, 0.25]
    probs= np.array(list(counts.values()), dtype=float)/ len(s)
    #apply shannon entropy formula 
    #measures how unpredictable the character distribution is
    entropy= -(probs*np.log2(probs)).sum()

    #low values -> low entropy 
    #high values -> high entropy 
    return float(entropy)

#pass entropy per sessions 
pw_by_session= (
    sess_events.dropna(subset=["password"])
    .groupby("session")["password"].apply(list).to_dict()                          
)

def session_pass_entropy(session: str)-> Tuple[float, float]:
    pws= pw_by_session.get(session, [])
    if not pws:
        return(0.0, 0.0)
    ent= [shannon_entropy(pw) for pw in pws if isinstance(pw, str)]
    if not ent:
        return (0.0, 0.0)
    return (float(np.mean(ent)), float(np.max(ent)))

pw_stats= session_df["session"].apply(session_pass_entropy)
session_df["pw_entropy_mean"]= [t[0] for t in pw_stats]
session_df["pw_entropy_max"]= [t[1] for t in pw_stats]

#rates
session_df["cmds_per_min"]= session_df["cmd_count"]/(session_df["duration"]/60.0+1e-9)
session_df["fails_per_min"]= session_df["login_fail"]/(session_df["duration"]/60.0+1e-9)

#boolean flags 
session_df["has_successful_login"]= (session_df["login_success"]>0).astype(int)
session_df["has_any_commands"]= (session_df["cmd_count"]>0).astype(int)

session_df[["session", "duration", "cmd_count", "cmds_per_min","login_fail", "fails_per_min", "pw_entropy_mean", "has_successful_login"]].head(10)

### GeoIP Location
Enable if using geolocation for country, city, lan, lon

In [ ]:
if GEOIP:
    import geoip2.database 
    reader= geoip2.database.Reader(str(GEOIP_MMDB))

    #maps ip to location metadata and return none if missing or failed lookups
    def geoip_lookup(ip: str) -> Dict[str, Any]:
        if not isinstance(ip,str) or not ip:
            return{"country": None, "city": None, "lat": None, "lon": None}
        try:
            resp= reader.city(ip)
            return{
                "country": getattr(resp.country, "iso_code", None),
                "city": getattr(resp.city, "name", None),
                "lat": getattr(resp.location, "latitude", None),
                "lon": getattr(resp.location, "longitude", None),
            }
        except Exception:
            return {"country": None, "city": None, "lat": None, "lon": None}
    
    geo= session_df["src_ip"].apply(geoip_lookup).apply(pd.Series)
    session_df= pd.concat([session_df, geo.add_prefix("geo_")], axis=1)

session_df.head(5)

### ML Feature Table 
Constructs final dataset for training
- One row per session
- Cleaned with not empty values
- Numeric behavioral features

In [ ]:
feature_cols_numeric= [
    "duration",
    "cmd_count",
    "unique_cmds",
    "cmds_per_min",
    "login_success",
    "login_fail",
    "fails_per_min",
    "unique_users",
    "unique_pass",
    "pw_entropy_mean",
    "pw_entropy_max",
    "has_success_login",
    "has_any_commands",
]

#if geolocation is enabled
if GEOIP:
    feature_cols_numeric+= ["geo_lat", "geo_lon"]

features_df= session_df[["session", "src_ip", "session_start", "session_end"]+feature_cols_numeric].copy()
features_df.replace([np.inf, -np.inf], np.nan, inplace=True)
features_df.fillna(0, inplace=True)
features_df.head()


### Visualizations
Graphs to demonstrate attacker behavior patterns 
- Session duration
- Commands per session
- Login failures per session
- Mean password entropy per session
- Top IP sources


In [ ]:
#plot histogram for numeric session features and visualize feature distribution 
def hist(series: pd.Series, title:str, bins: int= 50):
    plt.figure()
    #drop missing values
    plt.hist(series.dropna().values, bins=bins)
    plt.title(title)
    plt.xlabel(series.name)
    plt.ylabel("count")
    plt.show()

#distribution plots
hist(session_df["duration"], "Session duration (s)")
hist(session_df["cmd_count"], "Commands per session")
hist(session_df["login_failures"], "Login failures per session")
hist(session_df["pw_entropy_mean"], "Mean password entropy per session")

#top source ips
top_ips= session_df["src_ip"].value_counts().head(15)
plt.figure()
plt.bar(top_ips.index.astype(str), top_ips.values)
plt.title("Top source IPs by session count")
plt.xticks(rotation=70, ha="right")
plt.ylabel("sessions")
plt.show()

### Save files for backend and training

In [ ]:
#file paths for outputs
events_out= OUTPUT_DIR/"cowrie_events.csv"
sessions_out= OUTPUT_DIR/"cowrie_sessions.csv"
features_out= OUTPUT_DIR/"cowrie_features.csv"

events_df.to_csv(events_out, index= False)
session_df.to_csv(sessions_out, index= False)
features_df.to_csv(features_out, index= False)

print("Saved Files:")
print("-", events_out)
print("-", sessions_out)
print("-", features_out)
